In [9]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB # New Import
from sklearn.metrics import classification_report, accuracy_score

# --- MOCK DATA CREATION ---
# (Skipping the mock data creation block for brevity, assuming the data is now loaded)

# --- 1. Data Loading and Parsing ---

def parse_conllu_data(file_path, language_code):
    """Parses a custom CoNLL-U file and extracts sentences and their labels."""
    data = []
    current_sentence = ""
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith('# text = '):
                    if current_sentence:
                        data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = line.split('=', 1)[1].strip()
                elif not line and current_sentence:
                    data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = ""
            if current_sentence:
                 data.append({'text': current_sentence, 'label': language_code})
                 
    except FileNotFoundError:
        return pd.DataFrame() 
        
    return pd.DataFrame(data).drop_duplicates()

data_dir = 'data'
# The languages list should be based on the actual files you have.
# Using 'de' and 'en' here for demonstration continuity.
languages = ['be', 'de', 'en', 'es', 'fr', 'it', 'ko', 'pt', 'ru', 'ta']

all_data = []
print("--- Parsing Data ---")
for lang_code in languages:
    filename = f'output_{lang_code}.conllu'
    file_path = os.path.join(data_dir, filename)
    df = parse_conllu_data(file_path, lang_code)
    if not df.empty:
        print(f"Parsed file: {filename} with {len(df)} sentences.")
        all_data.append(df)
    
df_combined = pd.concat(all_data, ignore_index=True)

# --- 2. Sampling and Data Preparation (100 Samples) ---

N_SAMPLES = 6660000  
total_samples = len(df_combined)

if total_samples == 0:
    print("\nError: No data was loaded.")
else:
    print(f"\nTotal sentences loaded: {total_samples}")

    if total_samples >= N_SAMPLES:
        df_sampled = df_combined.sample(n=N_SAMPLES, random_state=42)
        print(f"Successfully sampled {N_SAMPLES} sentences.")
    else:
        df_sampled = df_combined
        print(f"Warning: Only {total_samples} available, using all data instead of {N_SAMPLES}.")

    le = LabelEncoder()
    df_sampled['label_id'] = le.fit_transform(df_sampled['label'])
    label_map = dict(zip(le.classes_, le.transform(le.classes_)))
    print("Label Mapping:", label_map)

    X_train, X_test, y_train, y_test = train_test_split(
        df_sampled['text'], 
        df_sampled['label_id'], 
        test_size=0.2, 
        random_state=42,
        stratify=df_sampled['label_id']
    )

    vectorizer = TfidfVectorizer(max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print(f"\nTraining set size: {X_train_vec.shape}")
    print(f"Testing set size: {X_test_vec.shape}")

    # --- 3. Model Training and Evaluation (Multinomial Naive Bayes) ---
    
    print("\nTraining the Multinomial Naive Bayes Model...")
    
    # Instantiate the Naive Bayes model
    model = MultinomialNB() 
    model.fit(X_train_vec, y_train)

    # Make Predictions
    y_pred = model.predict(X_test_vec)

    # Evaluate the Model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nModel Accuracy: {accuracy:.4f}")

    # Detailed Classification Report
    print("\nClassification Report (Language ID -> Language Code):")
    class_names = le.classes_
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

--- Parsing Data ---
Parsed file: output_be.conllu with 256841 sentences.
Parsed file: output_de.conllu with 665234 sentences.
Parsed file: output_en.conllu with 617835 sentences.
Parsed file: output_es.conllu with 1010430 sentences.
Parsed file: output_fr.conllu with 666444 sentences.
Parsed file: output_it.conllu with 335780 sentences.
Parsed file: output_ko.conllu with 406643 sentences.
Parsed file: output_pt.conllu with 728576 sentences.
Parsed file: output_ru.conllu with 1759991 sentences.
Parsed file: output_ta.conllu with 221603 sentences.

Total sentences loaded: 6669377
Successfully sampled 6660000 sentences.
Label Mapping: {'be': np.int64(0), 'de': np.int64(1), 'en': np.int64(2), 'es': np.int64(3), 'fr': np.int64(4), 'it': np.int64(5), 'ko': np.int64(6), 'pt': np.int64(7), 'ru': np.int64(8), 'ta': np.int64(9)}

Training set size: (5328000, 1000)
Testing set size: (1332000, 1000)

Training the Multinomial Naive Bayes Model...

Model Accuracy: 0.8726

Classification Report (Lan

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import MultinomialNB # New Import
from sklearn.metrics import classification_report, accuracy_score

# --- MOCK DATA CREATION ---
# (Skipping the mock data creation block for brevity, assuming the data is now loaded)

# --- 1. Data Loading and Parsing ---

def parse_conllu_data(file_path, language_code):
    """Parses a custom CoNLL-U file and extracts sentences and their labels."""
    data = []
    current_sentence = ""
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line.startswith('# text = '):
                    if current_sentence:
                        data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = line.split('=', 1)[1].strip()
                elif not line and current_sentence:
                    data.append({'text': current_sentence, 'label': language_code})
                    current_sentence = ""
            if current_sentence:
                 data.append({'text': current_sentence, 'label': language_code})
                 
    except FileNotFoundError:
        return pd.DataFrame() 
        
    return pd.DataFrame(data).drop_duplicates()

data_dir = 'data'
# The languages list should be based on the actual files you have.
# Using 'de' and 'en' here for demonstration continuity.
languages = ['be', 'de', 'en', 'es', 'fr', 'it', 'ko', 'pt', 'ru', 'ta']

all_data = []
print("--- Parsing Data ---")
for lang_code in languages:
    filename = f'output_{lang_code}.conllu'
    file_path = os.path.join(data_dir, filename)
    df = parse_conllu_data(file_path, lang_code)
    if not df.empty:
        print(f"Parsed file: {filename} with {len(df)} sentences.")
        all_data.append(df)
    
df_combined = pd.concat(all_data, ignore_index=True)

# --- 2. Sampling and Data Preparation (100 Samples) ---

N_SAMPLES = 6600000
total_samples = len(df_combined)

if total_samples == 0:
    print("\nError: No data was loaded.")
else:
    print(f"\nTotal sentences loaded: {total_samples}")

    if total_samples >= N_SAMPLES:
        df_sampled = df_combined.sample(n=N_SAMPLES, random_state=42)
        print(f"Successfully sampled {N_SAMPLES} sentences.")
    else:
        df_sampled = df_combined
        print(f"Warning: Only {total_samples} available, using all data instead of {N_SAMPLES}.")

    le = LabelEncoder()
    df_sampled['label_id'] = le.fit_transform(df_sampled['label'])
    label_map = dict(zip(le.classes_, le.transform(le.classes_)))
    print("Label Mapping:", label_map)

    X_train, X_test, y_train, y_test = train_test_split(
        df_sampled['text'], 
        df_sampled['label_id'], 
        test_size=0.2, 
        random_state=42,
        stratify=df_sampled['label_id']
    )

    vectorizer = TfidfVectorizer(max_features=1000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    print(f"\nTraining set size: {X_train_vec.shape}")
    print(f"Testing set size: {X_test_vec.shape}")

    # --- 3. Model Training and Evaluation (Multinomial Naive Bayes) ---
    
    print("\nTraining the Multinomial Naive Bayes Model...")
    
    # Instantiate the Naive Bayes model
    model = MultinomialNB() 
    model.fit(X_train_vec, y_train)

    # Make Predictions
    y_pred = model.predict(X_test_vec)

    # Evaluate the Model
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nModel Accuracy: {accuracy:.4f}")

    # Detailed Classification Report
    print("\nClassification Report (Language ID -> Language Code):")
    class_names = le.classes_
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

--- Parsing Data ---
Parsed file: output_be.conllu with 256841 sentences.
Parsed file: output_de.conllu with 665234 sentences.
Parsed file: output_en.conllu with 617835 sentences.
Parsed file: output_es.conllu with 1010430 sentences.
Parsed file: output_fr.conllu with 666444 sentences.
Parsed file: output_it.conllu with 335780 sentences.
Parsed file: output_ko.conllu with 406643 sentences.
Parsed file: output_pt.conllu with 728576 sentences.
Parsed file: output_ru.conllu with 1759991 sentences.
Parsed file: output_ta.conllu with 221603 sentences.

Total sentences loaded: 6669377
Successfully sampled 6600000 sentences.
Label Mapping: {'be': np.int64(0), 'de': np.int64(1), 'en': np.int64(2), 'es': np.int64(3), 'fr': np.int64(4), 'it': np.int64(5), 'ko': np.int64(6), 'pt': np.int64(7), 'ru': np.int64(8), 'ta': np.int64(9)}

Training set size: (5280000, 1000)
Testing set size: (1320000, 1000)

Training the Multinomial Naive Bayes Model...

Model Accuracy: 0.8726

Classification Report (Lan